# 《金玉良炎》公衛資料分析與 AI Code 協作

**學員版 Google Colab｜標準教學資料**

> 完全虛構聲明：本 notebook 與資料中的朝代、人物、疾病、病原、檢驗、地點、食品、事件與數值均為合成教學設定，不對應真實歷史、真實作品情節或真實醫學資料。

本 notebook 刻意不預載答案。你會沿固定流程請 Gemini 產生 Python，再用分母、原始計數、病例定義與資料限制檢查程式。

## 使用方式

1. 先取得 `jinyuliang_student_release_1.zip`。
2. 在 Google Colab 依序執行儲存格；找不到本機資料時會提示上傳 zip。
3. 只有在老師宣布時，才上傳 Release 2 與 Release 3。
4. 每次問 Gemini 前，先寫清楚：分析問題、分析單位、join key、分母、輸出、驗證檢查。

In [ ]:
from pathlib import Path
import io
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, norm

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

def find_release_dir(release_name):
    """Find local project data; otherwise ask for a release zip in Colab."""
    candidates = [
        Path.cwd() / "student" / release_name,
        Path.cwd().parent / "student" / release_name,
        Path("/content/jinyuliang") / "student" / release_name,
    ]
    marker = "person_line_list.csv" if release_name == "release_1" else "README.md"
    for candidate in candidates:
        if (candidate / marker).exists():
            return candidate

    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"找不到 {release_name}。請從專案根目錄執行，或在 Colab 上傳對應 zip。"
        ) from exc

    print(f"請上傳 {release_name} 的 zip 檔。")
    uploaded = files.upload()
    zip_name, zip_bytes = next(iter(uploaded.items()))
    destination = Path("/content/jinyuliang")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        archive.extractall(destination)
    matches = list(destination.rglob(marker))
    if not matches:
        raise FileNotFoundError(f"{zip_name} 中找不到 {marker}")
    return matches[0].parent

RELEASE_1 = find_release_dir("release_1")
print("Release 1:", RELEASE_1)

In [ ]:
people = pd.read_csv(RELEASE_1 / "person_line_list.csv", dtype={"person_id": "string"})
exposures = pd.read_csv(RELEASE_1 / "food_exposure.csv", dtype={"person_id": "string"})
menu = pd.read_csv(RELEASE_1 / "menu_by_site.csv")
dictionary = pd.read_csv(RELEASE_1 / "data_dictionary.csv")

datetime_columns = [
    "meal_start_datetime", "meal_end_datetime", "survey_datetime",
    "symptom_onset_datetime", "symptom_end_datetime",
]
for column in datetime_columns:
    people[column] = pd.to_datetime(people[column], errors="coerce")

display(pd.DataFrame({
    "table": ["people", "exposures", "menu", "dictionary"],
    "rows": [len(people), len(exposures), len(menu), len(dictionary)],
    "columns": [people.shape[1], exposures.shape[1], menu.shape[1], dictionary.shape[1]],
}))

## 1. 資料載入、schema 與分母

**先填規格**

- 分析問題：名冊、食品暴露與菜單是否能正確連結？
- 分析單位：人／人×食品／供餐組合。
- 分母：參與人數與回覆率用完整名冊；食品項目缺失率只在問卷回覆者中計算。

**請 Gemini**

> 請用 pandas 檢查三張表的列數、主鍵、外鍵、每人食品列數、binary 欄位允許值與缺失比例。不要把缺失補成 0。每個檢查請輸出 observed value，並用 assert 標出不符合規格之處。先用文字重述你採用的分母。

**人工驗收**：`person_id` 是否唯一？每人食品項目數是否一致？合併前後列數是否符合分析單位？

In [ ]:
# TODO：把 Gemini 產生的資料品質檢查程式貼在這裡。
# 最少要輸出：roster_n、response_n、food_rows_per_person、item_missing_rate。

## 2. 將工作手冊翻成病例定義

**請 Gemini**

> 請依《疾病工作手冊》建立 `classify_cases()`。輸入為 person line list；時間基準使用個人用餐結束；建立 `hours_after_meal`、`primary`、`sensitive`、`strict`。時間窗必須是函式參數。未回覆問卷者三個病例欄位都保持 pandas nullable Int64 的 NA，不得編成 0。請逐段解釋括號與布林條件。

**人工驗收**：抽查至少 3 人；確認病例欄位只有 0／1／NA；確認主要病例不使用檢驗資料。

In [ ]:
# TODO：在這裡實作 classify_cases()，再建立 cases 與 case_summary。
# 建議參數：primary_window=(4, 36), sensitive_window=(3, 48), strict_window=(5, 30)

## 3. Person／Time／Place

**請 Gemini**

> 使用 person_line_list.csv 內的 primary、site_id、role_group、meal_session、symptom_onset_datetime 與 meal_end_datetime。先依 site_id、role_group、meal_session 計算 classifiable_n、cases、attack_rate；再建立 hours_after_meal，輸出主要病例潛伏期的中位數、IQR、範圍與落在手冊 5–28 小時主要區間的比例。最後寫一個 `plot_epi_curve(data, case_col='primary', bin_hours=4)`，可改 1、2、4、6 小時分箱。圖標題必須顯示病例定義與 bin 寬度，三圖共用時間範圍。空白 symptom_onset_datetime 保留為缺失值。

**人工驗收**：各角色、場地與梯次都要同時列病例數與可分類分母；各場地病例數加總應等於總病例數；每種分箱的柱高總和應相同；潛伏期起點必須使用每人的 meal_end_datetime。

In [ ]:
# TODO：Person／Place 摘要表。

In [ ]:
# TODO：畫 1、2、4、6 小時分箱的流行曲線，並檢查病例總數不變。

## 4. 十項食品的 2×2、發病率、OR、CI 與 Fisher test

**請 Gemini**

> 請對 food_exposure 長表中的每個 food_id 建立 2×2 表。只使用該食品 reported_consumed 與 primary 都已知者。輸出 a=暴露病例、b=暴露非病例、c=未暴露病例、d=未暴露非病例、n_complete、兩組發病率、OR、95% CI、雙尾 Fisher exact p。零格時使用 0.5 修正並標記。最後依 OR 排序，畫 log x 軸 forest plot。不要只回傳 p 值。

**人工驗收**：任選兩項手算 `a*d/(b*c)`；每列確認 `a+b+c+d=n_complete`；不同食品的分母可以不同。

In [ ]:
# TODO：建立可重用的 odds_ratio_ci() 與 two_by_two()。

In [ ]:
# TODO：逐項食品分析並畫 forest plot。

## 5. 共食與多變項 logistic regression

**請 Gemini**

> 把食品暴露由長表 pivot 成每人一列，計算食品暴露 Pearson/phi 相關矩陣並畫 heatmap。根據粗分析與共食結構選兩項候選食品。先依 site_id 建立主要候選食品的分層 2×2 表，保留各層 a、b、c、d，計算層內 OR 與 Mantel–Haenszel pooled OR／95% CI；再建立 `primary ~ candidate_1 + candidate_2 + C(site_id)` 的 statsmodels logistic regression。模型只用完整案例，輸出 nobs、病例數、參考組、收斂狀態、OR、95% CI 與 p 值，最後比較粗 OR、MH OR 與調整 OR。十項食品先以粗分析篩選；溫度作為環境證據另外呈現。

**人工驗收**：各場地四格與分母是否完整？若層內出現零格，是否保留原始四格並說明估計方式？模型分母是否清楚？參照場地是哪一個？是否收斂？粗、MH 與調整效果的變化是否符合共食結構？

In [ ]:
# TODO：暴露共食 heatmap。

In [ ]:
# TODO：選定兩項候選食品後，用 statsmodels 擬合調整模型。
# 如環境缺少 statsmodels，可先執行：!pip -q install statsmodels

## 6. 暴露缺失敏感度分析

**請 Gemini**

> 對主要候選食品比較三個情境：complete case、暴露缺失全視為 0、暴露缺失全視為 1。只能改 reported_consumed，不得填補 primary。三個情境都重用同一個 two_by_two()，輸出 n 與原始四格計數、OR、95% CI。

**人工驗收**：兩個極端情境的 n 應相同；complete case n 較小；確認沒有改到病例狀態。

In [ ]:
# TODO：三情境缺失敏感度分析與 forest plot。

## 7A. Release 2：病人檢驗（老師宣布後才執行）

**請 Gemini**

> 讀取 Release 2 的 lab_results.csv。以 sample_id 為檢驗列、person_id 為受檢人，分別回答「做了幾項檢驗」與「有幾人受檢」。連結 onset 後計算相對發病採檢時間，按 test_name 與採檢時間窗摘要陽性率。依工作手冊解釋陰性結果，不要用陰性排除臨床病例。

In [ ]:
# 老師宣布後，把 Release 2 zip 上傳；也可在專案根目錄直接執行。
# RELEASE_2 = find_release_dir("release_2")
# labs_r2 = pd.read_csv(RELEASE_2 / "lab_results.csv", dtype={"person_id": "string"})
# TODO：區分檢驗列數與受檢人數，並分析採檢時間。

## 7B. Release 3：食品、分型與環境證據（老師宣布後才執行）

**請 Gemini**

> 讀取 Release 3 的完整 lab_results.csv 與 environment_log.csv。建立「統計暴露證據、病人檢驗、食品檢驗／分型、環境條件、限制」矩陣。用 person_id 連病人，用 food_id/site_id/serving_batch_id 連食品與環境。特別檢查任何陽性 target 是否真的會造成金玉良炎，不得只看 result='positive'。

In [ ]:
# RELEASE_3 = find_release_dir("release_3")
# labs_r3 = pd.read_csv(RELEASE_3 / "lab_results.csv", dtype={"person_id": "string"})
# environment = pd.read_csv(RELEASE_3 / "environment_log.csv")
# TODO：比對 subtype、批次、溫度與候膳時間，建立證據矩陣。

## 8. 五句調查摘要

請完成五句，每句都能回指程式輸出：

1. **事件與分母**：完整名冊、可分類人數與回覆率。
2. **病例與型態**：病例定義、病例數、時間／地點分布。
3. **暴露分析**：主要候選食品的食用／未食用發病率、粗 OR 與 CI。
4. **調整與三角驗證**：共食調整後結果、病人／食品／環境證據。
5. **限制**：缺失、檢驗限制、觀察性資料與不可外推範圍。

避免使用「單一 p 值證明病因」、「第一位通報者是污染來源」或「陰性即未感染」等語句。

## 9. 匯出完整調查報告（HTML）

把每一幕實際產生的圖存到 `outputs/figures/`，再於下方 `report_notes` 填入自己的判讀。執行後會產生一個內嵌圖片、可單獨開啟的 HTML 報告；也可用瀏覽器列印成 PDF。

正式報告只收錄你的圖表與判讀，不會自動帶入講師的示範答案。

In [ ]:
from pathlib import Path
from datetime import datetime
import base64, html

report_notes = {
    'event_summary': '請填寫事件、調查啟動依據與完整名冊分母。',
    'case_definition': '請填寫主要病例定義、病例數與替代定義結果。',
    'person_time_place': '請填寫流行曲線與場地侵襲率的判讀。',
    'exposure_analysis': '請填寫食品別侵襲率、粗 OR 與調整後效果。',
    'evidence': '請分開說明人體檢驗、群聚病因與食品媒介證據。',
    'limitations': '請填寫缺失、回憶、採檢與觀察性資料限制。',
}

report_figures = {
    'Person／Time／Place': 'outputs/figures/epi_curve_4h.png',
    '食品粗分析': 'outputs/figures/food_forest_plot.png',
    '粗與調整效果': 'outputs/figures/adjusted_or_plot.png',
    '缺失敏感度分析': 'outputs/figures/missing_sensitivity.png',
}

def image_data_uri(path):
    path = Path(path)
    if not path.exists():
        return None
    mime = 'image/svg+xml' if path.suffix.lower() == '.svg' else 'image/png'
    return f'data:{mime};base64,' + base64.b64encode(path.read_bytes()).decode()

sections = [
    ('事件摘要', 'event_summary'), ('病例定義', 'case_definition'),
    ('Person／Time／Place', 'person_time_place'), ('暴露分析', 'exposure_analysis'),
    ('檢驗與環境證據', 'evidence'), ('調查限制', 'limitations'),
]
body = []
for title, key in sections:
    body.append(f'<section><h2>{html.escape(title)}</h2><p>{html.escape(report_notes[key])}</p>')
    uri = image_data_uri(report_figures.get(title, ''))
    if uri:
        body.append(f'<figure><img src=\"{uri}\" alt=\"{html.escape(title)}\"><figcaption>{html.escape(title)}</figcaption></figure>')
    body.append('</section>')

report_html = f'''<!doctype html><html lang=\"zh-Hant\"><head><meta charset=\"utf-8\">
<title>千秋宴金玉良炎群聚調查奏報</title><style>
body{{max-width:900px;margin:48px auto;padding:0 32px;color:#202824;background:#f8f4eb;font-family:Arial,'Microsoft JhengHei',sans-serif;line-height:1.8}}
header{{padding:36px;border-block:3px double #9b4a35;text-align:center}} h1,h2{{font-family:serif}} h2{{margin-top:42px;color:#24594e}}
figure{{margin:28px 0;padding:18px;background:white;border:1px solid #d8cbb6}} img{{max-width:100%;height:auto}} figcaption{{color:#6f736d;font-size:12px}}
.fiction{{color:#9b4a35;font-weight:bold}} @media print{{body{{background:white;margin:0}} section{{break-inside:avoid}}}}
</style></head><body><header><p class=\"fiction\">完全虛構之公共衛生教學案例</p><h1>千秋宴金玉良炎群聚調查奏報</h1><p>產製時間：{datetime.now():%Y-%m-%d %H:%M}</p></header>
{''.join(body)}</body></html>'''

output_path = Path('jinyuliang_investigation_report.html')
output_path.write_text(report_html, encoding='utf-8')
print(f'已產生：{output_path.resolve()}')
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    from IPython.display import display, FileLink
    display(FileLink(str(output_path)))